# LocalFold

One cell. What appears below is the whole of localfold.org - sequences,
ligands, modified residues, templates, every model, the viewer, the plots and
the downloads - and its Fold button runs on **this runtime's GPU**.

**Runtime → Change runtime type → T4 GPU** before you run it.

---

How it works, because the arrangement is not the obvious one: a page's
JavaScript runs where the page is SHOWN, so a LocalFold in this cell would
normally fold on your laptop with the runtime idle. Instead the runtime serves
the page and folds for it - LocalFold's own `web/app.js` in a headless Chrome
there, on the card Colab lent you - and the page in the cell asks it. Same
origin, no tunnel, nothing to paste.

Your browser still draws: the viewer, the plots, the sequence strip. That is
what a browser is for.

In [ ]:
#@title LocalFold { display-mode: "form" }
#@markdown Run this cell. The page appears below - the whole of localfold.org,
#@markdown with sequences, ligands, modified residues, templates, every model,
#@markdown the viewer, the plots and the downloads - and its **Fold button
#@markdown folds on this runtime's GPU**, not on your laptop.
#@markdown
#@markdown Set **Runtime → Change runtime type → T4 GPU** first. The install
#@markdown takes about two minutes the first time and nothing after that.
height = 1200  #@param {type:"slider", min:600, max:2000, step:50}
branch = "main"  #@param {type:"string"}

import json, os, queue, subprocess, sys, threading, time

PORT, REPO = 8710, '/content/localfold'
SETUP = r"""
set -e
# 🔴 THE USERSPACE HALF OF THE DRIVER, MATCHED TO THE KERNEL HALF. A runtime
# ships the kernel module and nvidia-smi, not the Vulkan ICD a browser needs -
# and a Chrome that asks for Vulkan and does not find it is handed SwiftShader,
# the CPU renderer, which folds and means nothing. Measured on the first real
# runtime: vendor 'google', architecture 'swiftshader'.
DRIVER=$(nvidia-smi --query-gpu=driver_version --format=csv,noheader 2>/dev/null | cut -d. -f1)
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y libvulkan1 vulkan-tools dbus > /dev/null 2>&1
[ -n "$DRIVER" ] && apt-get -qq install -y "libnvidia-gl-${DRIVER}" > /dev/null 2>&1 || true
if ! command -v google-chrome > /dev/null; then
  wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
  apt-get -qq install -y ./google-chrome-stable_current_amd64.deb > /dev/null 2>&1
fi
if [ -d REPO_PATH ]; then
  git -C REPO_PATH fetch -q origin BRANCH && git -C REPO_PATH checkout -q FETCH_HEAD
else
  git clone -q --depth 1 --branch BRANCH https://github.com/sokrypton/localfold REPO_PATH
fi
"""
with open('/content/_localfold_setup.sh', 'w') as handle:
    handle.write(SETUP.replace('REPO_PATH', REPO).replace('BRANCH', branch))
print('setting up…')
subprocess.run(['bash', '/content/_localfold_setup.sh'], check=True)

# 🔴 THE SERVICE SERVES THE PAGE AND DRIVES THE FOLD, which is what makes one
# cell enough: the frame below loads index.html FROM it, so the page and the
# thing that folds are the same origin and there is nothing to paste, no CORS
# and no tunnel. See tools/colab_backend.py.
def _drain(stream, sink):
    for line in iter(stream.readline, ''):
        sink.put(line.rstrip())

if not globals().get('_localfold_service'):
    _localfold_service = subprocess.Popen(
        [sys.executable, 'tools/colab_backend.py', '--port', str(PORT)],
        cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    _lines = queue.Queue()
    threading.Thread(target=_drain, args=(_localfold_service.stdout, _lines), daemon=True).start()
    TOKEN, ADAPTER = None, None
    _deadline = time.time() + 300
    while time.time() < _deadline and TOKEN is None:
        try:
            said = _lines.get(timeout=5)
        except queue.Empty:
            continue
        if said.startswith('BACKEND '):
            answer = json.loads(said[len('BACKEND '):])
            TOKEN, ADAPTER = answer['token'], answer['gpu']
        elif 'rror' in said:
            print(said)
    if TOKEN is None:
        raise SystemExit('the fold service did not start; run this cell again')

# 🔴 THE ONE LINE WORTH READING BEFORE ANY FOLD. 'nvidia' and an architecture
# is the card; 'swiftshader' or 'llvmpipe' is the CPU wearing its clothes, and
# every fold after that is a CPU fold that looks exactly like success.
print('folding on:', ADAPTER.get('vendor'), ADAPTER.get('architecture'),
      '| f16', ADAPTER.get('shaderF16'), '| subgroup matrix', ADAPTER.get('subgroupMatrix'))
if 'swiftshader' in json.dumps(ADAPTER).lower() or 'llvmpipe' in json.dumps(ADAPTER).lower():
    print('*** that is the CPU renderer, not the card - check the runtime type ***')

from google.colab.output import eval_js
from IPython.display import HTML, display

base = eval_js('google.colab.kernel.proxyPort(%d)' % PORT)
# `?backend=colab` is the page being told which machine folds; `t` is the
# token the service requires of every request, and it rides in the URL because
# a page cannot be handed a header by whoever framed it.
page = '%sindex.html?backend=colab&t=%s' % (base, TOKEN)
display(HTML(
    '<p style="font:13px system-ui;margin:4px 0">'
    '<a href="%s" target="_blank" rel="noopener">open in its own tab</a>'
    ' &middot; folds on the runtime</p>'
    '<iframe src="%s" width="100%%" height="%d" '
    'allow="webgpu; clipboard-write" '
    'style="border:1px solid #e5e7eb;border-radius:8px"></iframe>' % (page, page, height)))